# 9. XGBoost

## 9.1 概念

回顾：已经学过 GBDT，知道 GBDT 是一棵树一棵树接力，每棵树去拟合上一轮的残差（负梯度）。

学了 GBDT，它很好用，但是它有三个天生的短板，XGBoost 就是专门解决这三个短板的升级版 GBDT。

GBDT 每一棵树都在拟合负梯度，可它从来不问“这棵树是不是太复杂了”：叶子越多、叶子的输出越大，越容易把噪声也学进去，只能靠学习率和早停在外面兜着。2014 年，华盛顿大学的博士生陈天奇写了 XGBoost（eXtreme Gradient Boosting），把对树的惩罚直接写进每一棵树要最小化的目标，又用二阶导数让每一步走得更准，还让树自己学会把缺失值放到哪一边。它在当年的数据竞赛里拿下了大量冠军，今天仍是风控、营销、竞赛里最常用的模型之一。

总结：GBDT 只管“把损失降下去”，不太管树会不会学疯、学过拟合；XGBoost 是在“降低损失”的同时，主动给树加约束，并且用上二阶导数、自带缺失值处理、速度更快。

## 9.2 GBDT 主要三个问题

### 1. 没有内置惩罚，容易过拟合

GBDT 在分裂节点的时候，只看：这么分裂之后，损失能不能变小。它不会去问：这棵树是不是叶子太多了？单个叶子的输出值是不是太大了？如果数据里有噪声、异常点，它会拼命分裂，造很多叶子，强行把噪声也记住。GBDT 只能靠外面的手段兜底：限制树深度、调小学习率、早停，属于“事后管控”，不是写在目标里面的约束。

类比：GBDT 像一个只追求考试分数的学生，为了提分疯狂死记硬背（连错题、干扰信息一起背），老师只能靠规定最长学习时间、及时叫停（早停）来防止他学偏。XGBoost 是在打分规则里直接写好：背的知识点条目越多（叶子多）扣分项越高、单次改动幅度太大（叶子输出大）也要扣分，从根源上抑制死记硬背。

### 2. 只用一阶导数信息

GBDT 找分裂点，只看一阶导数（负梯度 / 残差），相当于只知道“往哪个方向走能变好”，不知道走这一步，变化有多快。

- 一阶导数 = 坡度方向；
- 二阶导数 = 坡度的陡峭程度。

只知道方向，不知道坡度陡缓，步子就容易踩不准。

类比：下山，一阶导数只告诉你往哪边下坡；二阶导数告诉你这个坡是很陡还是很平缓，帮你判断一步迈多大最合适。

### 3. 速度慢、不会处理缺失值

sklearn 里的 GBDT，找分裂点是单线程、挨个阈值慢慢试；而且一旦遇到缺失值，直接报错，必须我们手动填充。

XGBoost 自带稀疏感知分裂，遇到缺失值，模型自己学习：缺失样本应该分到左边分支还是右边分支，不用我们人工填充，同时还有并行优化，速度快很多。

XGBoost 的三个改进：目标函数加正则惩罚、分裂计算用一二阶导数、原生支持缺失值 + 工程加速。

## 9.3 目标函数：损失 + 正则化

第 m 棵树 $f_m$ 要最小化的是：

$$\text{Obj} = \sum_{i=1}^{n} L\left(y_i, F_{m-1}(x_i) + f_m(x_i)\right) + \Omega(f_m), \qquad \Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

理解：Obj = 【原来的损失项】 + 【对这棵新树的惩罚项】。

- 前面求和的 L：就是 GBDT 本来就有的损失，衡量预测和真实标签差多少；
- $\Omega(f_m)$：正则惩罚，XGBoost 新加的，GBDT 没有。
  - T：这棵树一共有多少叶子；γ：每多一片叶子，就扣 γ 分。作用：不让树随便分裂，叶子不能乱多；
  - $w_j$：第 j 个叶子的输出值；λ 是 L2 正则系数，惩罚叶子输出不要太大。作用：叶子的预测输出不能猛一下跳很大，限制输出幅度，防止单叶子对结果影响爆炸。

例子：我们现在打分，不光看你做题扣了多少分（损失 L），还要额外扣分：

- 你整理的笔记条目越多（叶子数量 T），每条扣 γ；
- 你写的结论改动幅度越大（w 越大），额外扣 λ 的惩罚。

所以模型在分裂的时候，会主动权衡：增加这个叶子带来的收益，能不能覆盖掉新增的惩罚，收益不够就不分裂。

区分：GBDT 的惩罚是“外面的枷锁”（max_depth、early_stop）；XGBoost 的惩罚是写进目标函数里，模型每一次分裂都会主动考虑代价。

## 9.4 从目标函数到相似度分数

XGBoost 最核心的推导，我们不用硬抠泰勒展开数学，抓逻辑：

泰勒二阶展开：把损失函数在当前预测值附近展开到二阶，每个样本算出两个数：

- $g_i$：一阶导数（梯度）；
- $h_i$：二阶导数（海森）。

同一叶子里的所有样本，输出都是同一个 $w_j$。我们把叶子内部所有样本的 g 加起来 = $G_j$，所有 h 加起来 = $H_j$。目标函数就拆成一片片叶子单独的二次函数。

对叶子输出 $w_j$ 求导找最优值，得到叶子最好的输出：

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

再代回去，得到这棵树能达到的最小目标：

$$\text{Obj}^* = -\frac{1}{2}\sum_{j=1}^{T}\frac{G_j^2}{H_j+\lambda} + \gamma T$$

这里的 $\frac{G_j^2}{H_j+\lambda}$，就是相似度分数 similarity score。

理解：这个分数代表把这一组样本放在同一个叶子里，能帮我们减少多少损失。

- 如果叶子里面样本的梯度方向一致（全都偏大、全都偏小）：G 很大，相似度分数高，说明这群样本放一起很合适，收益很高；
- 如果里面一部分梯度正、一部分梯度负，互相抵消：G 很小，分数很低，这群样本鱼龙混杂，放一起收益很小。

类比：一个小组（叶子），组员的意见（梯度）方向一致，合力大，这个小组价值高；组员意见互相打架，合力抵消，小组价值很低。

## 9.5 分裂增益 Gain

现在我们要做分裂：把一个叶子，切分成左叶子 L、右叶子 R。分裂之后，目标函数减少多少，就是分裂带来的收益（增益 Gain）：

$$\text{Gain} = \frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}$$

然后我们还要付出代价：多了一片叶子，要扣 γ（库里面的 gamma，min_split_loss）。

判断规则：

- 如果 $\text{Gain} > \gamma$：切，收益大于新增叶子的惩罚，划算；
- 如果 $\text{Gain} \le \gamma$：不切，收益不够，不值得多开一个叶子，防止无意义分裂。

对比：

GBDT 分裂只看：分裂之后损失降没降，没有这个“开叶子的固定成本”，只要损失降一点点就会无脑分裂，很容易碎成很多小叶子、记住噪声。

XGBoost 每一次分裂，都同时考虑：L2 惩罚 λ、新增叶子代价 γ，这两个正则参数直接参与分裂决策。

## 9.6 XGBoost 是什么样的树

XGBoost 每一棵树，不是直接去预测房价，而是去预测【残差】，也就是上一轮预测错了多少；相似度分数，用来衡量把一堆样本放在同一个叶子节点好不好。

例子：现在只有 5 套房子，想预测房子单价。

- 一开始我们啥信息都没有，最简单粗暴的预测：全部房子都猜成平均单价，这个平均值就是 f0，也就是 XGBoost 这一轮的起点；
- 残差 r = 真实单价 − 平均单价：代表【平均值猜得差了多少】。比如一套房子真实 3.25，平均值是 2.182，残差 = 3.25−2.182=1.068，意思是：平均值低估了它 1.068；
- 相似度分数公式：$\frac{(\sum r)^2}{n+\lambda}$。
  - $\sum r$：这个节点里面所有样本残差加起来；
  - n：这个节点里面有多少套房子；
  - λ：正则惩罚项，防止树太复杂、过拟合（相当于给打分加一个“扣分”）。

相似度分数越高 → 把这堆样本放在同一个叶子，预测效果越好。

当我们用全部样本的平均值当起点的时候，所有残差加起来一定等于 0！$\sum r = r_1+r_2+r_3+r_4+r_5=0$，所以 $(\sum r)^2=0$，根节点相似度分数直接等于 0。

意思就是：如果这棵树一刀都不分，全部样本挤在同一个叶子，这棵树没有学到任何东西，损失不会下降。所以 XGBoost 必须做分裂，切分节点，才可以拿到大于 0 的相似度增益，降低误差。

In [1]:
# 导入numpy数值计算库，pandas表格库
import numpy as np
import pandas as pd

# 设置随机种子3，保证每次运行，随机生成的数据一模一样
rng = np.random.default_rng(3)
# 生成60个0~30之间的房龄，并且从小到大排序
age = np.sort(rng.uniform(0, 30, 60))
# 按照一个隐藏公式生成60套房子的单价：房龄越大，价格总体越低，再加一点正弦波动+随机噪声
price = 3.2 - 0.06 * age + 0.25 * np.sin(age / 3) + rng.normal(0, 0.18, 60)

# 另外再准备验证集数据
rv = np.random.default_rng(33)
age_v = rv.uniform(0, 30, 300)
price_v = 3.2 - 0.06 * age_v + 0.25 * np.sin(age_v / 3) + rv.normal(0, 0.18, 300)

# 从60套房子里面，挑选下标2、15、30、45、57这5套，作为我们本节课的5个样本
pick = [2, 15, 30, 45, 57]
# 取出这5套的房龄，保留1位小数
x5, y5 = np.round(age[pick], 1), np.round(price[pick], 2)

# f0：起点预测值，也就是这5套房子单价的平均值（XGBoost回归初始值）
f0 = y5.mean()
# 计算残差 r = 真实价格y5 - 初始预测平均值f0
r5 = y5 - f0

# 打印5套房子的房龄
print("房龄：", x5.tolist())
# 打印5套房子真实单价
print("单价：", y5.tolist())
# 打印起点平均值，保留3位小数
print("起点（平均单价）：", round(f0, 3))
# 打印残差，保留3位小数
print("残差：", np.round(r5, 3).tolist())

# 循环两种λ取值：λ=0（不做正则惩罚），λ=1（加上正则惩罚）
for lam in [0, 1]:
    # 套用相似度分数公式：(残差总和)^2 / (样本数量n + λ)，保留4位小数输出
    print(f"λ = {lam}：根节点的相似度分数 (Σr)²/(n + λ) = {r5.sum() ** 2 / (5 + lam):.4f}")

房龄： [2.6, 8.8, 15.5, 22.0, 27.9]
单价： [3.25, 2.79, 1.78, 1.95, 1.14]
起点（平均单价）： 2.182
残差： [1.068, 0.608, -0.402, -0.232, -1.042]
λ = 0：根节点的相似度分数 (Σr)²/(n + λ) = 0.0000
λ = 1：根节点的相似度分数 (Σr)²/(n + λ) = 0.0000


## 9.7 阈值

刚发现：不分裂的时候整棵树（根节点）相似度分数是 0，啥都学不到。

XGBoost 长树的核心：遍历所有可以切的位置，挨个算切这一刀之后能拿到多少“收益（增益 gain）”，选收益最大的那一刀，树就长出第一个分叉。

例子：5 套房子房龄已经排好序：[2.6, 8.8, 15.5, 22.0, 27.9]。

n 个有序样本，相邻两个中间，一共可以切 n−1 刀，也就是 4 个候选分割点：

- 第 1 刀：2.6 和 8.8 的中间：(2.6+8.8)/2 = 5.7；
- 第 2 刀：8.8 和 15.5 中间：(8.8+15.5)/2 = 12.15；
- 第 3 刀：15.5 和 22 中间：(15.5+22.0)/2 = 18.75；
- 第 4 刀：22 和 27.9 中间：(22.0+27.9)/2 = 24.95。

规则：

- 每一刀把房子分成左边一组、右边一组；
- 分别算左边叶子相似度分数、右边叶子相似度分数；
- 增益 = 左相似度 + 右相似度 − 原来根节点相似度；
- 增益越大，代表这一刀分割之后，整体拟合效果提升越多，误差下降越多。

相似度分数公式：$\frac{(\sum r)^2}{n+\lambda}$，r 是这一组样本的残差，n 是本组样本数量，λ=1。

In [2]:
# 目的：在4个候选阈值上挨个切一刀，计算每一刀的分裂增益，λ固定为1
lam = 1.0    # 正则超参数λ，本节课固定λ=1，用来防止分数虚高，抑制过拟合

# 定义一个函数sim，输入一组残差r，输出这一组的相似度分数
# sim = lambda r: (残差总和)^2 / (本组样本数量 + λ)
sim = lambda r: r.sum() ** 2 / (len(r) + lam)

# 计算全部候选分割阈值：相邻房龄两两取中点
# x5[:-1]：取除最后一个之外所有房龄；x5[1:]：取除第一个之外所有房龄
# 两两相加除以2，得到4个切分点
cuts = (x5[:-1] + x5[1:]) / 2

# 循环遍历每一个候选分割阈值c
for c in cuts:
    # L = x5 <= c：布尔数组，标记哪些房子落在左边（房龄≤当前阈值c）
    L = x5 <= c
    # r5[L] 左边样本残差；r5[~L] 右边样本残差（~代表取反）
    # 增益 = 左相似度 + 右相似度 - 根节点相似度
    gain = sim(r5[L]) + sim(r5[~L]) - sim(r5)
    # 格式化打印输出，对齐方便看
    print(f"房龄 ≤ {c:>5}：左 {sim(r5[L]):.4f} + 右 {sim(r5[~L]):.4f} − 根 {sim(r5):.4f} = 增益 {gain:.4f}")

房龄 ≤   5.7：左 0.5703 + 右 0.2281 − 根 0.0000 = 增益 0.7984
房龄 ≤ 12.15：左 0.9363 + 右 0.7022 − 根 0.0000 = 增益 1.6386
房龄 ≤ 18.75：左 0.4058 + 右 0.5410 − 根 0.0000 = 增益 0.9468
房龄 ≤ 24.95：左 0.2172 + 右 0.5429 − 根 0.0000 = 增益 0.7600


总结：增益最大的是 1.6386，因此应该在【8.8, 15.5】之间切一刀。XGBoost 就会从这两个数之间做第一次分裂，这棵树第一个分叉就长出来了！

## 9.8 叶子的输出与 λ

前面我们算出了在哪一刀分裂收益最大，但分裂之后，每个叶子节点要输出一个预测值，这个值不是简单的残差平均值，公式是：

$$w = \frac{\sum r}{n+\lambda}$$

- w：叶子输出（叶子权重）；
- $\sum r$：本组残差之和；
- n：叶子里样本数量；
- λ：L2 正则系数。

λ 相当于一个“阻尼拉力”，它会把叶子的预测值往 0 的方向拽：

- λ=0：没有拉力，输出就是残差的平均值 $\frac{\sum r}{n}$，完全跟着本组样本走；
- λ 越大，拉力越强；样本越少的叶子，被拽得越明显，防止少量几个样本就让叶子输出一个很极端的值，用来抑制过拟合。

In [3]:
# 目的：使用上一节增益最大的分割点best，计算左右叶子的输出值，对比不同λ带来的效果
# 遍历所有候选分割的增益，找到增益最大的那个阈值，作为最优分割点best
best = cuts[int(np.argmax([sim(r5[x5 <= c]) + sim(r5[x5 > c]) for c in cuts]))]
# L：布尔标记，房龄小于等于best的样本，分到左叶子
L = x5 <= best
print("第一刀：房龄 ≤", best)

# 依次测试 λ=0、λ=1、λ=10，观察正则的影响
for lam_ in [0, 1, 10]:
    # 左叶子输出：左残差总和 / (左样本数量 + λ)
    left_w = r5[L].sum() / (L.sum() + lam_)
    # 右叶子输出：右残差总和 / (右样本数量 + λ)
    right_w = r5[~L].sum() / ((~L).sum() + lam_)
    # L.sum()：左叶子样本数量；(~L).sum()：右叶子样本数量
    print(f"λ = {lam_:>2}：左叶（{L.sum()} 套）输出 {left_w:.4f}，右叶（{(~L).sum()} 套）输出 {right_w:.4f}")

第一刀：房龄 ≤ 12.15
λ =  0：左叶（2 套）输出 0.8380，右叶（3 套）输出 -0.5587
λ =  1：左叶（2 套）输出 0.5587，右叶（3 套）输出 -0.4190
λ = 10：左叶（2 套）输出 0.1397，右叶（3 套）输出 -0.1289


λ 的作用：L2 正则，把叶子输出往 0 收缩。样本越少的叶子，受 λ 影响越大！

## 9.9 γ 剪枝

λ 是控制叶子输出大小；γ（gamma）是分裂门槛，XGBoost 里名字叫 min_split_loss：这一刀分裂带来的增益，必须大于 γ，这一刀才值得留下来。

XGBoost 论文原版剪枝策略是【先长满、再从下往上剪】，不是一边长一边停！

流程：先按照 max_depth，使劲把树长到最深；然后从最底层的子节点往根节点反向检查：

1. 如果当前这一刀的增益 ≤ γ → 剪掉这一刀，左右两片叶子合并回一个节点；
2. 特殊规则：只要下面还有一刀没有被剪掉，上面父节点这一刀哪怕增益不够 γ，也不能剪！

类比：γ 就像“开店门槛”：开分店（分裂），新增赚的利润（增益）必须大于门槛 γ，分店才值得开。先一口气把所有分店都开出来（长到最大深度），然后从最下层分店开始复盘：

- 下层分店赚的钱不够门槛 → 关掉（剪枝）；
- 只要下层还有分店活着，上层总店哪怕利润不高，也不能直接关掉总店。

区分两个容易混淆的生长方式：

1. 先长满再剪（论文原版，本节代码实现）：先全部长出来，自底向上剪；可以捕捉“底层分裂收益很大、上层分裂收益一般”的情况（异或类数据）；
2. 分裂前停止（提前停止）：长的时候直接判断，增益不够就不分裂；上层收益低就直接停，底层再好也长不出来。

In [4]:
# 目的：复现XGBoost论文原版剪枝：先把树长到深度2，再自下而上剪枝
# λ=0，单独观察γ的作用；增益 = 左sim + 右sim - 父sim，和xgboost gamma同尺度

# 定义相似度函数，λ=0，sim = (残差总和)^2 / 样本数量
sim0 = lambda r: r.sum() ** 2 / len(r)

# 函数：输入一组残差r_、对应特征x_，找出这一组里面最优分割点c、以及对应的增益gain
def best_cut(r_, x_):
    # 生成所有候选分割阈值：相邻样本中点
    cs = (x_[:-1] + x_[1:]) / 2
    # 遍历所有阈值，计算每个阈值对应的（分割点，增益），选出增益最大的一组返回
    return max(((c, sim0(r_[x_ <= c]) + sim0(r_[x_ > c]) - sim0(r_)) for c in cs), key=lambda z: z[1])

# 对全部5个样本，找根节点最优分割点、增益
root = best_cut(r5, x5)
# Lm：布尔标记，分到根节点左分支的样本
Lm = x5 <= root[0]

# 对左分支这一组样本，继续找最优分割（长出左孩子）
left = best_cut(r5[Lm], x5[Lm])
# 对右分支这一组样本，继续找最优分割（长出右孩子）
right = best_cut(r5[~Lm], x5[~Lm])

# 打印三层的分割信息：根、左子节点、右子节点
print(f"根：房龄 ≤ {root[0]}，增益 {root[1]:.4f}；左孩子：≤ {left[0]}，增益 {left[1]:.4f}；右孩子：≤ {right[0]}，增益 {right[1]:.4f}")

# 遍历4组不同γ值，观察剪枝结果
for gamma in [0, 0.2, 0.5, 2.5]:
    # 判断：左孩子是否保留：左增益 > γ 才留；增益≤γ就剪掉
    keep_l, keep_r = left[1] - gamma > 0, right[1] - gamma > 0
    # 核心规则：只要左孩子保留 或者 右孩子保留 或者 根自己增益>γ → 根节点这一刀必须保留
    # 翻译：只要下层还有一刀没剪，上层根节点哪怕增益不够γ，也不能剪！
    keep_root = keep_l or keep_r or root[1] - gamma > 0
    print(f"γ = {gamma:<4}：左孩子{'留' if keep_l else '剪'}，右孩子{'留' if keep_r else '剪'}，根{'留' if keep_root else '剪（整棵树只剩一片叶子）'}")

根：房龄 ≤ 12.15，增益 2.3408；左孩子：≤ 5.7，增益 0.1058；右孩子：≤ 24.95，增益 0.3504
γ = 0   ：左孩子留，右孩子留，根留
γ = 0.2 ：左孩子剪，右孩子留，根留
γ = 0.5 ：左孩子剪，右孩子剪，根留
γ = 2.5 ：左孩子剪，右孩子剪，根剪（整棵树只剩一片叶子）


剪枝顺序：从下往上，先检查底层子节点，再看上层父节点，不是从上往下！

核心铁律：下层只要有一刀保留，上层父节点这一刀就不能剪。

λ 和 γ 的对比：

- λ：相似度、叶子输出的时候生效，压缩叶子权重，减少单样本带来极端预测；λ 变大，所有增益都会变小，更容易被 γ 剪掉；
- γ：分裂门槛，决定这一刀能不能存在；γ 越大，树越简单、分支越少，抑制过拟合。

## 9.10 手写 XGBoost

核心主线一句话：【一轮一轮种树，每一棵树都去拟合上一轮剩下的“误差残差”；每棵树内部：算一阶导数 g、二阶导数 h → 找最优分裂点 → 长到最大深度 → 自下 γ 剪枝 → 叶子输出公式 $w = -G/(H+\lambda)$ → 乘学习率 η 加到总预测上】。

类比：我们要预测房价，一开始只会猜一个固定均价 $f_0$；

- 第 1 棵树：专门去学习「哪些房子比均价贵、哪些便宜」，算出误差；
- 第 2 棵树：不去直接学房价，专门学上一轮预测还剩下的误差；
- 第 3 棵继续学剩下误差……一棵树一棵树接力修正；
- 每一棵树内部，就是我们前面的全部内容：相似度、增益、λ 正则、γ 剪枝、min_child_weight（最小 H 和）。

In [5]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# ===================== 函数 build：递归构建单棵XGB树，包含【先长满、再自底向上剪枝】逻辑 =====================
# x：当前节点这批样本的特征（房龄）
# g：这批样本的一阶梯度g
# h：这批样本的二阶海森h
# lam：正则系数λ
# gamma：剪枝门槛γ（min_split_loss）
# depth：允许最大深度
# mcw：min_child_weight，子节点H总和下限
# d：当前深度，递归用，默认从0开始
def build(x, g, h, lam, gamma, depth, mcw, d=0):
    G, H = g.sum(), h.sum()                          # G：当前节点梯度总和；H：当前节点海森总和
    node = {"w": -G / (H + lam)}                     # 先默认这个节点是叶子节点，叶子输出公式 w = -G/(H+λ)
    if d >= depth:                                   # 已经到达最大深度，不能继续分裂，直接返回叶子
        return node

    # 1. 把样本按特征x从小到大排序，方便遍历候选分割点
    o = np.argsort(x)                                # 得到排序下标
    xs, gs, hs = x[o], g[o], h[o]                    # 排序后的x、g、h

    # 2. 前缀和，快速计算分割后左半部分G、H
    GL, HL = np.cumsum(gs)[:-1], np.cumsum(hs)[:-1]  # GL：左区间梯度累加；HL：左区间海森累加；去掉最后一个（全部样本）
    # ok条件：
    # ① 前后x值不一样，相同特征值不能切；
    # ② 左子H≥mcw；③ 右子H≥mcw；满足才是合法分割点
    ok = (xs[1:] != xs[:-1]) & (HL >= mcw) & (H - HL >= mcw)
    if not ok.any():                                 # 没有任何合法分割点，直接返回叶子
        return node

    # 3. 计算每一个候选分割点的增益（和XGB论文一致，不带1/2）
    gain = GL ** 2 / (HL + lam) + (G - GL) ** 2 / (H - HL + lam) - G ** 2 / (H + lam)
    gain = np.where(ok, gain, -np.inf)               # 不合法分割点增益设为负无穷，不会被选中
    k = int(np.argmax(gain))                         # 选出增益最大的分割点下标

    if gain[k] <= 0:                                 # 最优分割增益≤0，分裂之后不会变好，不分裂，返回叶子
        return node

    # 4. 确定分割阈值，递归构建左、右子树
    t = (xs[k] + xs[k + 1]) / 2                      # 分割阈值：两个相邻样本中点
    L = x <= t                                       # 布尔掩码：左分支样本
    # 把分割信息、增益、左右子树放进node字典
    node.update(t=t, gain=float(gain[k]),
                L=build(x[L], g[L], h[L], lam, gamma, depth, mcw, d + 1),
                R=build(x[~L], g[~L], h[~L], lam, gamma, depth, mcw, d + 1))

    # ========== 核心：自底向上剪枝逻辑（论文原版剪枝） ==========
    # 判断：左孩子、右孩子都已经是叶子（没有t键，代表不能再分裂）
    # 并且这一刀的增益 ≤ γ → 剪掉这一刀，左右叶子合并回单个叶子
    if "t" not in node["L"] and "t" not in node["R"] and node["gain"] - gamma <= 0:
        return {"w": node["w"]}
    return node                                      # 不需要剪，返回带分裂的节点

# ===================== 单棵树预测函数：递归遍历树，输入一个x，返回叶子输出w =====================
def predict_tree(node, x):
    # 没有t，说明是叶子，直接返回权重w
    if "t" not in node:
        return node["w"]
    # 有分割阈值t，判断走左分支还是右分支，递归下去
    return predict_tree(node["L"], x) if x <= node["t"] else predict_tree(node["R"], x)

# ===================== 手写XGB主训练循环：多棵树Boosting循环 =====================
# x：特征，y：标签；n_trees树数量；eta学习率；lam λ；depth最大深度；gamma γ；mcw min_child_weight
def my_xgb(x, y, n_trees, eta, lam, depth, gamma=0.0, mcw=0.0):
    F = np.full(len(y), y.mean())                    # 初始预测：全部样本先用均值 f0
    trees = []                                       # 用来保存训练好的每一棵树
    for m in range(n_trees):                         # 循环：依次训练n_trees棵树
        g, h = F - y, np.ones(len(y))                # 平方损失：g=F-y，h恒等于1
        t = build(x, g, h, lam, gamma, depth, mcw)   # 训练这一轮的树，拟合当前梯度g、h
        trees.append(t)                              # 把树存入列表
        # 更新总预测：F = F + η * 这棵树对每个样本的预测值
        F = F + eta * np.array([predict_tree(t, v) for v in x])
    return y.mean(), trees                           # 返回初始均值f0 + 全部树列表

# ===================== 整体预测函数：把所有树的结果累加起来 =====================
def my_predict(model, x, eta):
    f0, trees = model
    base = f0
    # 每一棵树的预测值相加，乘以学习率η，叠加到基础值上
    add = eta * np.array([sum(predict_tree(t, v) for t in trees) for v in x])
    return base + add

# ---------------------- 下面是训练、对比sklearn直方图GBDT ----------------------
# 这里age、price是前面课程的60套房数据（房龄、房价）
mine = my_xgb(age, price, n_trees=30, eta=0.3, lam=1.0, depth=2, gamma=0.0, mcw=0.0)
# sklearn HistGradientBoostingRegressor，对齐一模一样的超参数
hgb = HistGradientBoostingRegressor(
    max_iter=30,                # 树的数量
    learning_rate=0.3,          # 学习率η
    max_depth=2,                # 树最大深度
    max_leaf_nodes=None,
    min_samples_leaf=1,
    l2_regularization=1.0,      # 对应我们代码里的λ
    early_stopping=False        # 关闭提前停止，保证和我们逐棵训练完全对齐
).fit(age.reshape(-1, 1), price)
# 打印对比指标（age_v、price_v是验证集）
print("自己写的，验证 MSE：", round(np.mean((price_v - my_predict(mine, age_v, 0.3)) ** 2), 4))
print("sklearn 的，验证 MSE：", round(np.mean((price_v - hgb.predict(age_v.reshape(-1, 1))) ** 2), 4))
print("60套房上两者的预测最多差：", float(np.abs(my_predict(mine, age, 0.3) - hgb.predict(age.reshape(-1, 1))).max()))

自己写的，验证 MSE： 0.0432
sklearn 的，验证 MSE： 0.0432
60套房上两者的预测最多差： 2.4782926821842466e-09


自己写的 XGBoost 与 sklearn `HistGradientBoostingRegressor` 的验证 MSE 都是 0.0432，60 套房上的预测最多只差约 $2.48 \times 10^{-9}$，说明手写逻辑与官方直方图 GBDT 对齐。

## 9.12 引入官方提供的 XGBoost

### 9.12.1 安装 XGBoost

`pip install xgboost`

### 9.12.2 XGBoost 实战：信贷违约预测

直接用工业界原版 xgb 库，做真实信贷风控案例：预测客户会不会违约。

背景：银行审批贷款，模型就是一个经验丰富的审核员，反复用很多棵简单小树投票；每棵树从客户的年龄、收入、学历这些信息里找规律，判断这个人会不会欠钱不还。

In [ ]:
# 导入时间模块
import time
# 导入官方xgboost库
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
# 导入AUC计算工具，风控、二分类不平衡场景最常用指标
from sklearn.metrics import roc_auc_score

# 读取信贷csv表格，一行代表一个贷款客户
df = pd.read_csv('../data/信贷数据.csv')

# 定义连续数值特征：可以直接加减比较的数字
num = ["年龄", "月收入", "负债收入比", "额度使用率", "逾期次数", "工作年限"]
# 定义类别特征：是分类标签，没有大小关系，比如学历：本科/硕士，住房：自有/租房
cat = ["学历", "住房", "职业"]

# 开启enable_categorical之后，XGB可以原生支持类别特征，不用手动做独热编码OneHot
df[cat] = df[cat].astype('category')
# 划分训练集、测试集
X_train, X_test, y_train, y_test = train_test_split(
    df[num+cat],
    df['是否违约'],
    test_size=0.2,
    stratify=df['是否违约'],
    random_state=42
)

# 实例化XGB分类器（二分类：违约/不违约）
clf = xgb.XGBClassifier(
    n_estimators=2000,        # 最大允许训练2000棵树；搭配早停，一般不会跑到2000棵
    learning_rate=0.05,       # 学习率，也叫eta；每一棵树只修正一小部分误差。步子小，更稳，不容易过拟合；步子小就要多堆树
    max_depth=3,              # 单棵树最大深度只有3层，小树！工业界XGB很少用很深的树，浅树+大量树是标配
    min_child_weight=1,       # 叶子节点最小样本权重和；调大之后，叶子不能包含太少样本，对噪声更不敏感
    gamma=0,                  # 分裂增益门槛，增益大于gamma才允许分裂；0代表不设置额外门槛
    subsample=0.8,            # 行采样：每一棵树，随机抽取80%的训练样本去训练，增加随机性，抑制过拟合
    colsample_bytree=0.8,     # 列采样：每一棵树，随机挑选80%的特征来分裂，类似随机森林，防止某几个特征垄断预测
    reg_lambda=1.0,           # L2正则λ，和我们手写版本的reg_lambda一模一样，压制叶子权重，让预测更平滑
    tree_method='hist',       # 直方图算法：把连续特征分桶，不用遍历每一个分割点，速度快很多；同时是原生类别特征支持的前提
    enable_categorical=True,  # 开启原生类别特征支持！category列不用我们自己独热编码，XGB内部自动找最优分割点
    eval_metric='auc',        # 早停监控指标：监控验证集AUC，风控任务优先看AUC
    early_stopping_rounds=50, # 早停：连续50轮（50棵树），验证集AUC不再上涨，立刻停止训练，不再继续生成树，节省时间、防止过拟合
    random_state=0,           # 模型内部随机种子，和前面切分数据的42相互独立，保证实验可复现
)

# 开始训练模型
clf.fit(
    X_train, y_train,
    # 给早停用的验证集，早停必须指定eval_set，用来每一轮计算AUC
    eval_set=[(X_test, y_test)],
    # 关闭逐轮打印日志；如果改成True，每一棵树都会打印AUC，2000轮会刷屏
    verbose=False
)

# .score默认返回准确率accuracy
print(clf.score(X_test, y_test))

# roc_auc_score
y_pred_prob = clf.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, y_pred_prob))

## 9.13 xgboost 原生接口

### 9.13.1 区分

- `XGBClassifier`：sklearn 包装接口，写法和 sklearn 其它模型（逻辑回归、随机森林）一致，上手简单，适合快速建模、网格搜索；
- `xgb.train` / `xgb.cv` / `DMatrix`：XGB 原生底层接口，速度更快、控制更细、工业调参、离线 5 折交叉验证的标准写法；风控、比赛里经常用。

类比：sklearn 包装版 = 自动挡汽车，一键开走；原生接口 = 手动挡，可以精细控制离合、档位，适合专业车手调极限性能。

案例：信贷违约预测。

In [ ]:
import xgboost as xgb

# 1. 构造 DMatrix —— XGBoost 的原生数据容器
# DMatrix 是XGB专门设计的内部数据结构，不是普通DataFrame/数组
# 好处：提前做分桶、缺失值预处理，训练的时候不用反复处理，速度更快；原生支持缺失值、类别特征
# enable_categorical=True 代表开启类别特征支持
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)

# 2. 参数字典 —— 原生API参数，和sklearn包装版参数名不完全一样
# 原生接口用字典传参，参数名和XGBClassifier有别名差异，不能混用
xgb_params = {
    # ---------- 任务目标配置 ----------
    "objective": "binary:logistic",  # 任务：二分类，输出0~1的概率（sigmoid）
                                      # 坑：单词拼写错（obejective）不会报错，会悄悄变成回归任务，结果完全错乱，很难排查
    "eval_metric": "auc",             # 评估指标：AUC，适合正负样本不平衡的信贷场景

    # ---------- 学习率、树结构 ----------
    "eta": 0.05,                      # eta = sklearn里面的learning_rate学习率
                                      # 每一棵树只修正一点点误差，步子小，模型更稳；eta小，一般需要更多树
    "max_depth": 3,                   # 单棵树最大深度3层，小树，抑制单棵树过拟合
    "min_child_weight": 1,            # 叶子节点样本权重和下限；调大，叶子不能放太少样本，对噪声更保守
    "gamma": 0,                       # 分裂增益门槛，增益大于gamma才允许分裂；0代表不限制
                                      # gamma越大，树越简单，很多分裂直接不做

    # ---------- 采样与正则（防过拟合） ----------
    "subsample": 0.8,                 # 行采样：每一棵树随机挑选80%样本训练，增加随机性
    "colsample_bytree": 0.8,          # 列采样：每一棵树随机挑选80%特征来分裂，类似随机森林
    "lambda": 1.0,                    # L2正则，对应sklearn里reg_lambda，压制叶子权重，让预测更平滑

    # ---------- 性能与复现 ----------
    "tree_method": "hist",            # 直方图分桶建树，速度快；同时是原生类别特征支持的必要条件
    "seed": 0,                        # 随机种子，固定采样、分桶的随机性，保证每次运行结果一样
}

# 3. xgb.cv：原生5折交叉验证
res = xgb.cv(
    xgb_params,                       # 上面定义好的参数字典
    dtrain,                           # 训练数据（DMatrix格式）
    num_boost_round=2000,             # 最多训练2000棵树，早停会提前终止
    nfold=5,                          # 5折交叉验证
    stratified=True,                  # 分层折：每一折里面违约客户比例和原始数据保持一致，不平衡分类必开
                                      # 回归任务没有分层，要关掉这个参数
    early_stopping_rounds=50,         # 早停：5折平均验证AUC，连续50棵树不再上涨，立刻停止训练
                                      # 注意：cv里的早停看的是5折的平均结果，不是单一验证集
    seed=0,                           # 控制折的划分、模型训练的随机种子，和params里seed相互独立
    verbose_eval=False,               # 关闭逐轮打印；改成True，每一轮都会打印5折训练、验证AUC
)

# 4. 解读返回结果res
# res是一个DataFrame表格，每一行代表新增一棵树之后的累计效果
# 列：
# train-auc-mean：5折训练集AUC平均值
# train-auc-std：5折训练AUC标准差，标准差越小，5个折效果越稳定
# test-auc-mean：5折验证集AUC平均值（我们最关心的指标）
# test-auc-std：5折验证AUC标准差

# idxmax() 找到验证集AUC最大值对应的行索引，也就是最优树的棵数
best_round = int(res['test-auc-mean'].idxmax())

# 打印最优轮数、对应的5折平均AUC
print(f"最佳轮数：{best_round}, 最佳CV AUC： {res['test-auc-mean'].max():.4f}")

# 打印最后几轮指标，观察AUC是不是已经平稳、不再上涨，确认早停生效
print(res.tail(5))

讲义运行输出：

最佳轮数：134, 最佳CV AUC: 0.7527

| | train-auc-mean | train-auc-std | test-auc-mean | test-auc-std |
| --- | --- | --- | --- | --- |
| 130 | 0.838982 | 0.003340 | 0.752567 | 0.021573 |
| 131 | 0.839298 | 0.003297 | 0.752625 | 0.021573 |
| 132 | 0.839665 | 0.003250 | 0.752535 | 0.021547 |
| 133 | 0.839800 | 0.003228 | 0.752617 | 0.021631 |
| 134 | 0.840185 | 0.003224 | 0.752683 | 0.021594 |